# 03 — Grafo de relacionamentos via `pessoas.parquet`

Não existe (ainda) um parquet de arestas pré-computado (ver
`docs/perf-plan-2026-05.md` §13.2). O padrão de 1 hop — "empresas que
compartilham uma pessoa com a empresa X" — é um self-join direto sobre
`pessoas.parquet` (ADR 0024), suficientemente barato em DuckDB/Ibis.

In [ ]:
import ficha_py
from ficha_py import _

con = ficha_py.connect_ia(month="2026-04")
pessoas = ficha_py.pessoas(con)

## Empresas conectadas a uma raiz via sócio/representante em comum

In [ ]:
cnpj_base_alvo = "00000001"

a = pessoas.filter(_.cnpj_base == cnpj_base_alvo)
b = pessoas

conectadas = (
    a.join(
        b,
        [a.cpf_mascarado == b.cpf_mascarado, a.nome_normalizado == b.nome_normalizado],
    )
    .filter(b.cnpj_base != cnpj_base_alvo)
    .select(cnpj_base=b.cnpj_base, papel=b.papel, via=a.nome_normalizado)
    .distinct()
)
conectadas.execute()

## Quando materializar um parquet de arestas

Só vale a pena pré-computar `(cnpj_a, cnpj_b, via_pessoa)` se travessia de
N hops (CTE recursiva) se mostrar lenta demais no DuckDB-WASM do frontend,
ou se for necessária análise de grafo global (componentes conexos, hubs).
Ver `docs/perf-plan-2026-05.md` §13.2 antes de implementar isso — não
desenhar sem medir primeiro.